# 📘 예외와 트레이스백

**예외 처리**는 프로그램의 오류를 우아하게 다루는 기법입니다.
`traceback` 모듈로 상세한 오류 정보를 얻을 수 있습니다.

**학습 목표:**
- try/except/else/finally 패턴
- 예외 계층과 사용자 정의 예외
- traceback 모듈로 오류 추적
- 예외 로깅과 체인드 예외

## 1. try/except 기본 패턴

파이썬은 `try/except`로 예외를 처리합니다.
`else`는 예외가 없을 때, `finally`는 항상 실행됩니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  예외 처리 구조                             │
# │  try:      → 예외 발생 가능 코드            │
# │  except:   → 예외 처리                      │
# │  else:     → 예외 없을 때 실행              │
# │  finally:  → 항상 실행 (정리 코드)           │
# └─────────────────────────────────────────┘

# 기본 예외 처리
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        return "0으로 나눌 수 없습니다"
    except TypeError as e:
        return f"타입 에러: {e}"
    else:
        return f"결과: {result}"
    finally:
        print("  (safe_divide 완료)")

print(safe_divide(10, 3))   # 정상
print(safe_divide(10, 0))   # ZeroDivisionError
print(safe_divide(10, "a")) # TypeError

# 여러 예외 동시 처리
try:
    value = int("abc")
except (ValueError, TypeError) as e:
    print(f"\n변환 에러: {type(e).__name__}: {e}")

## 2. 예외 계층과 사용자 정의 예외

모든 예외는 `Exception`의 서브클래스입니다.
사용자 정의 예외로 도메인 특화 에러를 만들 수 있습니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  예외 계층 (일부)                          │
# │  BaseException                             │
# │  ├── SystemExit                            │
# │  ├── KeyboardInterrupt                    │
# │  └── Exception                             │
# │      ├── ValueError                       │
# │      ├── TypeError                         │
# │      ├── KeyError                          │
# │      ├── IndexError                       │
# │      └── ...                               │
# └─────────────────────────────────────────┘

# 사용자 정의 예외
class InsufficientBalanceError(Exception):
    """잔액 부족 에러"""
    def __init__(self, balance, amount):
        self.balance = balance
        self.amount = amount
        super().__init__(f"잔액 {balance}원으로 {amount}원 출금 불가")

class BankAccount:
    def __init__(self, balance=0):
        self.balance = balance

    def withdraw(self, amount):
        if amount > self.balance:
            raise InsufficientBalanceError(self.balance, amount)
        self.balance -= amount
        return self.balance

account = BankAccount(1000)
print(f"초기 잔액: {account.balance}원")

account.withdraw(500)
print(f"출금 후: {account.balance}원")

try:
    account.withdraw(1000)
except InsufficientBalanceError as e:
    print(f"에러: {e}")
    print(f"잔액: {e.balance}원, 요청: {e.amount}원")

## 3. traceback 모듈

`traceback`은 예외 발생 위치와 호출 스택을 추적합니다.

In [ ]:
import traceback
import sys

# ┌─────────────────────────────────────────┐
# │  traceback 주요 함수                      │
# │  traceback.format_exc() → 에러 정보 문자열  │
# │  traceback.print_exc()  → 에러 정보 출력    │
# │  traceback.extract_stack() → 현재 스택    │
# └─────────────────────────────────────────┘

def inner_function():
    x = 1 / 0  # 의도적 에러

def middle_function():
    inner_function()

def outer_function():
    try:
        middle_function()
    except Exception:
        # 에러 정보를 문자열로
        error_info = traceback.format_exc()
        print("traceback.format_exc():")
        print(error_info)

outer_function()

# 정상 실행 시 스택 추출
print("\n현재 스택:")
stack = traceback.extract_stack()
for frame in stack[-3:]:
    print(f"  {frame.filename}:{frame.lineno} in {frame.name}")

## 🎯 연습 문제

1. 파일 읽기 시 `FileNotFoundError`와 `PermissionError`를 처리하는 코드를 작성하세요.
2. `NegativeValueError` 사용자 정의 예외를 만들고, 음수 입력 시 발생시키세요.
3. `traceback.format_exc()`를 사용해 예외를 파일에 기록하는 함수를 작성하세요.
4. `finally` 블록에서 파일을 닫는 패턴을 작성하세요.